In [45]:
import pandas as pd

matches_file = "matches.csv"
deliveries_file = "deliveries.csv"

matches_df = pd.read_csv(matches_file)
deliveries_df = pd.read_csv(deliveries_file)

matches_df.head(), deliveries_df.head()


(       id   season        city        date match_type player_of_match  \
 0  335982  2007/08   Bangalore  18-04-2008     League     BB McCullum   
 1  335983  2007/08  Chandigarh  19-04-2008     League      MEK Hussey   
 2  335984  2007/08       Delhi  19-04-2008     League     MF Maharoof   
 3  335985  2007/08      Mumbai  20-04-2008     League      MV Boucher   
 4  335986  2007/08     Kolkata  20-04-2008     League       DJ Hussey   
 
                                         venue                        team1  \
 0                       M Chinnaswamy Stadium  Royal Challengers Bangalore   
 1  Punjab Cricket Association Stadium, Mohali              Kings XI Punjab   
 2                            Feroz Shah Kotla             Delhi Daredevils   
 3                            Wankhede Stadium               Mumbai Indians   
 4                                Eden Gardens        Kolkata Knight Riders   
 
                          team2                  toss_winner toss_decision  \


In [46]:
team_mapping = {
    "Delhi Daredevils": "Delhi Capitals",
    "Deccan Chargers": "Sunrisers Hyderabad",
    "Rising Pune Supergiant": "Rising Pune Supergiants",
    "Kings XI Punjab": "Punjab Kings"
}
matches_df["winner"] = matches_df["winner"].replace(team_mapping)

venue_mapping = {
    "Punjab Cricket Association Stadium, Mohali": "Punjab Cricket Association IS Bindra Stadium",
    "M Chinnaswamy Stadium": "M. Chinnaswamy Stadium",
    "Feroz Shah Kotla": "Arun Jaitley Stadium"
}

matches_df["venue_canonical"] = matches_df["venue"].replace(venue_mapping)

matches_df = matches_df[["id", "winner", "venue_canonical"]]

matches_df.head()


,id,winner,venue_canonical
0,335982,Kolkata Knight Riders,M. Chinnaswamy Stadium
1,335983,Chennai Super Kings,Punjab Cricket Association IS Bindra Stadium
2,335984,Delhi Capitals,Arun Jaitley Stadium
3,335985,Royal Challengers Bangalore,Wankhede Stadium
4,335986,Kolkata Knight Riders,Eden Gardens


In [47]:
deliveries_df = deliveries_df[["match_id", "inning", "batting_team", "bowling_team", "over", "ball", "total_runs", "is_wicket"]]

deliveries_df["batting_team"] = deliveries_df["batting_team"].replace(team_mapping)
deliveries_df["bowling_team"] = deliveries_df["bowling_team"].replace(team_mapping)

deliveries_df["cum_runs"] = deliveries_df.groupby(["match_id", "inning"])["total_runs"].cumsum()
deliveries_df["cum_wickets"] = deliveries_df.groupby(["match_id", "inning"])["is_wicket"].cumsum()

deliveries_df["overs_completed"] = deliveries_df["over"] + (deliveries_df["ball"] / 6)

deliveries_df["current_run_rate"] = deliveries_df["cum_runs"] / deliveries_df["overs_completed"].replace(0, 1)

first_innings = deliveries_df[deliveries_df["inning"] == 1].groupby("match_id")["cum_runs"].max() + 1
first_innings = first_innings.reset_index().rename(columns={"cum_runs": "target"})

deliveries_df = deliveries_df.merge(first_innings, on="match_id", how="left")

deliveries_df["remaining_overs"] = 20 - deliveries_df["overs_completed"]
deliveries_df["required_run_rate"] = (deliveries_df["target"] - deliveries_df["cum_runs"]) / deliveries_df["remaining_overs"]
deliveries_df["required_run_rate"] = deliveries_df["required_run_rate"].replace([float("inf"), -float("inf")], 0).fillna(0)

deliveries_df = deliveries_df.merge(matches_df, left_on="match_id", right_on="id", how="left")
deliveries_df["win"] = (deliveries_df["batting_team"] == deliveries_df["winner"]).astype(int)

deliveries_df = deliveries_df[deliveries_df["inning"] == 2]

deliveries_df = deliveries_df[["match_id", "inning", "cum_runs", "cum_wickets", "current_run_rate",
                               "required_run_rate", "target", "batting_team", "bowling_team", "venue_canonical", "win"]]

deliveries_df.head()


<ipython-input-47-44a9e2c64f89>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  deliveries_df["batting_team"] = deliveries_df["batting_team"].replace(team_mapping)


,match_id,inning,cum_runs,cum_wickets,current_run_rate,required_run_rate,target,batting_team,bowling_team,venue_canonical,win
124,335982,2,1,0,6.0,11.193277,223,Royal Challengers Bangalore,Kolkata Knight Riders,M. Chinnaswamy Stadium,0
125,335982,2,2,0,6.0,11.237288,223,Royal Challengers Bangalore,Kolkata Knight Riders,M. Chinnaswamy Stadium,0
126,335982,2,2,0,4.0,11.333333,223,Royal Challengers Bangalore,Kolkata Knight Riders,M. Chinnaswamy Stadium,0
127,335982,2,3,0,4.5,11.379310,223,Royal Challengers Bangalore,Kolkata Knight Riders,M. Chinnaswamy Stadium,0
128,335982,2,4,0,4.8,11.426087,223,Royal Challengers Bangalore,Kolkata Knight Riders,M. Chinnaswamy Stadium,0


In [48]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pickle
import joblib

batting_encoder = LabelEncoder()
bowling_encoder = LabelEncoder()
venue_encoder = LabelEncoder()

deliveries_df["batting_team_encoded"] = batting_encoder.fit_transform(deliveries_df["batting_team"])
deliveries_df["bowling_team_encoded"] = bowling_encoder.fit_transform(deliveries_df["bowling_team"])

venue_column = "venue_canonical" if "venue_canonical" in deliveries_df.columns else "venue"
deliveries_df["venue_encoded"] = venue_encoder.fit_transform(deliveries_df[venue_column])

final_df = deliveries_df[["inning", "cum_runs", "cum_wickets", "current_run_rate",
                          "required_run_rate", "target", "batting_team_encoded",
                          "bowling_team_encoded", "venue_encoded", "win"]]

train_df, test_df = train_test_split(final_df, test_size=0.2, random_state=42, stratify=final_df["win"])

encoders = {"batting": batting_encoder, "bowling": bowling_encoder, "venue": venue_encoder}
with open("encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

joblib.dump(batting_encoder, "batting_encoder.pkl")
joblib.dump(bowling_encoder, "bowling_encoder.pkl")
joblib.dump(venue_encoder, "venue_encoder.pkl")

train_df.head()


,inning,cum_runs,cum_wickets,current_run_rate,required_run_rate,target,batting_team_encoded,bowling_team_encoded,venue_encoded,win
165910,2,116,3,8.385542,5.189189,148,0,1,0,1
48447,2,139,6,8.097087,7.764706,161,12,1,0,1
76799,2,67,2,6.483871,9.310345,157,7,12,23,0
241061,2,82,0,9.283019,8.597015,178,7,6,4,0
201744,2,47,0,7.833333,6.285714,135,0,14,47,1


In [50]:
# Keep 'venue_canonical_y' and drop 'venue_canonical_x'
deliveries_df.drop(columns=["venue_canonical_x"], inplace=True)
deliveries_df.rename(columns={"venue_canonical_y": "venue_canonical"}, inplace=True)

# Print to verify the change
print(deliveries_df.columns)


Index(['match_id', 'inning', 'cum_runs', 'cum_wickets', 'current_run_rate',
       'required_run_rate', 'target', 'batting_team', 'bowling_team', 'win',
       'batting_team_encoded', 'bowling_team_encoded', 'venue_encoded',
       'venue_canonical'],
      dtype='object')


In [51]:
venue_encoder = LabelEncoder()
deliveries_df["venue_encoded"] = venue_encoder.fit_transform(deliveries_df["venue_canonical"])


In [52]:
import pickle
import joblib

with open("venue_encoder.pkl", "wb") as f:
    pickle.dump(venue_encoder, f)

joblib.dump(venue_encoder, "venue_encoder.pkl")

print("Venue encoder saved successfully!")


Venue encoder saved successfully!


In [53]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "XGBoost": XGBClassifier()
}

for name, model in models.items():
    model.fit(train_df.drop(columns=["win"]), train_df["win"])
    print(f"{name} trained successfully!")


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression trained successfully!
Random Forest trained successfully!
XGBoost trained successfully!


In [54]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

for name, model in models.items():
    y_pred = model.predict(test_df.drop(columns=["win"]))
    print(f"\n===== {name} =====")
    print("Accuracy:", accuracy_score(test_df["win"], y_pred))
    print("Classification Report:\n", classification_report(test_df["win"], y_pred))
    print("Confusion Matrix:\n", confusion_matrix(test_df["win"], y_pred))



===== Logistic Regression =====
Accuracy: 0.7698516839635771
Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.73      0.75     12079
           1       0.77      0.80      0.78     13070

    accuracy                           0.77     25149
   macro avg       0.77      0.77      0.77     25149
weighted avg       0.77      0.77      0.77     25149

Confusion Matrix:
 [[ 8876  3203]
 [ 2585 10485]]

===== Random Forest =====
Accuracy: 0.9973756411785757
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     12079
           1       1.00      1.00      1.00     13070

    accuracy                           1.00     25149
   macro avg       1.00      1.00      1.00     25149
weighted avg       1.00      1.00      1.00     25149

Confusion Matrix:
 [[12052    27]
 [   39 13031]]

===== XGBoost =====
Accuracy: 0.9971370631039007
Classification Report:
       

In [55]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"]
}

log_reg = LogisticRegression()
grid_search = GridSearchCV(log_reg, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(train_df.drop(columns=["win"]), train_df["win"])

print("Best Logistic Regression Params:", grid_search.best_params_)
print("Best Logistic Regression Accuracy:", grid_search.best_score_)


Best Logistic Regression Params: {'C': 10, 'solver': 'lbfgs'}
Best Logistic Regression Accuracy: 0.7761650963327502


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [56]:
from xgboost import XGBClassifier

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 6, 9],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.7, 0.8, 1.0]
}

xgb = XGBClassifier()
grid_search = GridSearchCV(xgb, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(train_df.drop(columns=["win"]), train_df["win"])

print("Best XGBoost Params:", grid_search.best_params_)
print("Best XGBoost Accuracy:", grid_search.best_score_)


Best XGBoost Params: {'learning_rate': 0.2, 'max_depth': 9, 'n_estimators': 200, 'subsample': 1.0}
Best XGBoost Accuracy: 0.9993041186232304


In [57]:
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10]
}

rf = RandomForestClassifier()
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(train_df.drop(columns=["win"]), train_df["win"])

print("Best Random Forest Params:", grid_search.best_params_)
print("Best Random Forest Accuracy:", grid_search.best_score_)


Best Random Forest Params: {'max_depth': 30, 'min_samples_split': 2, 'n_estimators': 100}
Best Random Forest Accuracy: 0.9962124118824736
